# Detecção e diagnóstico de falhas em rolamentos — dados MFPT

ECM514 — Ciência de Dados, 2º semestre de 2026.

Máquinas que giram vibram de um jeito quando o rolamento está bom e de outro quando ele tem um defeito. Usamos gravações reais de vibração para responder duas perguntas: o rolamento tem algum problema (detecção) e, se tem, qual é o defeito (diagnóstico).

## 1. Os dados

Os dados da MFPT são gravações da vibração de um rolamento em bancada, em três situações: rolamento normal, defeito na pista externa e defeito na pista interna. Cada arquivo traz o sinal de vibração (em g), quantas medições foram feitas por segundo, a carga sobre o rolamento e a velocidade do eixo (25 voltas por segundo em todas).

Ao baixar o dataset, ele vem com uma pasta de análises, que só repete um dos arquivos de pista interna junto com os scripts de MATLAB da MFPT, e com uma pasta de exemplos de máquinas reais (uma bomba de óleo, um rolamento de velocidade intermediária e um rolamento planetário). Desconsideramos as duas: a cópia contaria a mesma gravação duas vezes, e os exemplos reais são outras máquinas, gravadas de outro jeito e sem a indicação de normal, pista externa ou pista interna. Ficamos com as 20 gravações de bancada.

As gravações normais foram feitas com 97.656 medições por segundo, e as de pista interna com 48.828. Se deixássemos assim, o modelo poderia separar as classes por esse detalhe da gravação, e não pelo defeito. Como 97.656 é exatamente o dobro de 48.828, passamos todas para 48.828: o sinal é filtrado e fica uma medição a cada duas. Isso corta a vibração acima de 24 mil ciclos por segundo, que as gravações de 48.828 nunca tiveram; as faixas que a própria MFPT usa nas análises dela ficam abaixo disso.

Os sinais também não oscilam exatamente em torno de zero, e esse deslocamento muda de uma sessão de gravação para outra. Como ele é do sensor e não do rolamento, tiramos a média de cada gravação.

In [ ]:
# Baixa o dataset do Kaggle; o kagglehub devolve a pasta onde os arquivos ficaram.
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub
from scipy.io import loadmat
from scipy.signal import decimate

path = kagglehub.dataset_download('emperorpein/mfpt-fault-datasets')

In [ ]:
# Lê só as 20 gravações de bancada (sem a pasta de análises e sem as máquinas reais) e monta uma
# tabela com uma linha por gravação. A classe sai do nome do arquivo, e a média de cada sinal é
# tirada porque ela vem do sensor, não do rolamento.
classes = {'baseline': 'normal', 'OuterRace': 'pista externa', 'InnerRace': 'pista interna'}

arquivos = sorted(f for f in glob.glob(os.path.join(path, '**', '*.mat'), recursive=True)
                  if '5 - Analyses' not in f and '6 - Real World' not in f)

linhas = []
for f in arquivos:
    b = loadmat(f, squeeze_me=True, struct_as_record=False)['bearing']
    nome = os.path.basename(f).replace('.mat', '')
    classe = next(c for chave, c in classes.items() if nome.startswith(chave))
    sinal = np.asarray(b.gs, dtype=float)
    linhas.append({'arquivo': nome, 'classe': classe, 'medicoes_por_seg': int(b.sr),
                   'carga': int(b.load), 'giro_hz': float(b.rate), 'media_original': sinal.mean(), 'sinal': sinal - sinal.mean()})

gravacoes = pd.DataFrame(linhas)
gravacoes['duracao_s'] = gravacoes.sinal.apply(len) / gravacoes.medicoes_por_seg
gravacoes.drop(columns='sinal').round(3)

In [ ]:
# Passa as gravações de 97.656 para 48.828 medições por segundo, para que a taxa de gravação não
# denuncie a classe. Depois resume cada classe com a força média da vibração (rms) de cada gravação.
taxa = 48828

gravacoes['sinal'] = [decimate(s, 2) if sr == 2 * taxa else s
                      for s, sr in zip(gravacoes.sinal, gravacoes.medicoes_por_seg)]
gravacoes['medicoes_por_seg'] = taxa
gravacoes['duracao_s'] = gravacoes.sinal.apply(len) / taxa
gravacoes['rms'] = gravacoes.sinal.apply(lambda s: np.sqrt(np.mean(s ** 2)))
gravacoes['pico'] = gravacoes.sinal.apply(lambda s: np.abs(s).max())

print(gravacoes[['arquivo', 'classe', 'carga', 'rms', 'pico']].round(2).to_string(index=False))
gravacoes.groupby('classe').agg(gravacoes=('arquivo', 'count'), segundos=('duracao_s', 'sum'),
                                rms_min=('rms', 'min'), rms_max=('rms', 'max'),
                                pico_min=('pico', 'min'), pico_max=('pico', 'max')).round(2)

In [ ]:
# Mostra 0,1 s de uma gravação de cada classe, na mesma escala, para ver a cara de cada defeito.
# Escolhemos gravações com carga 270 (ou a mais próxima, no caso da pista interna).
exemplos = ['baseline_1', 'OuterRaceFault_1', 'InnerRaceFault_vload_6']
n = int(0.1 * taxa)
t = np.arange(n) / taxa

fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True, sharey=True)
for ax, nome in zip(axes, exemplos):
    linha = gravacoes.set_index('arquivo').loc[nome]
    ax.plot(t, linha.sinal[:n], lw=0.6)
    ax.set_title(f'{linha.classe} ({nome}, carga {linha.carga})', fontsize=10)
    ax.set_ylabel('vibração (g)')
axes[-1].set_xlabel('tempo (s)')
plt.tight_layout()
plt.show()

### Resultados

Ficamos com 78 segundos de sinal, todos com 48.828 medições por segundo: 18 s de rolamento normal (3 gravações), 39 s com defeito na pista externa (10 gravações) e 21 s com defeito na pista interna (7 gravações).

O deslocamento do sensor confirma que ele vem da sessão de gravação, e não do defeito: fica entre -0,06 e -0,14 nas gravações de 97.656, e muda aos poucos, arquivo após arquivo, nas de 48.828 (de -0,19 a -0,16 na pista externa e de -0,23 a -0,19 na pista interna). Se não fosse tirado, o modelo poderia usá-lo para reconhecer a sessão.

A pista interna se separa fácil: força média da vibração (rms) entre 1,73 e 2,07 g e picos de 23 a 46 g, contra 0,74 a 0,75 g e picos de 3,4 a 3,8 g no rolamento normal. No gráfico ela aparece como pancadas fortes e espaçadas, de até 18 g nesse trecho.

A pista externa é o caso difícil. Nas três gravações feitas nas mesmas condições do normal (carga 270, mesma taxa de gravação), a força média fica em 0,55, 0,68 e 0,87 g, uma abaixo e outra acima do normal, e os picos (3,3 a 5,1 g) também se misturam com os do normal. Nem a força média nem o pico, sozinhos, separam essas gravações. As outras sete têm picos maiores (5 a 19 g), mas não existe gravação normal com essas cargas para comparar.

Limites dos dados:
- só há 3 gravações normais, todas com carga 270, gravadas nas mesmas condições das três primeiras de pista externa;
- a pista interna nunca foi gravada nas mesmas condições do normal, então medidas genéricas do sinal (força média, pico) podem estar captando a sessão de gravação junto com o defeito;
- a comparação mais justa que os dados permitem é normal contra as três gravações de pista externa com carga 270.

## 2. O ritmo de cada defeito

Um defeito no rolamento causa uma pancada cada vez que uma esfera passa por ele. Com as medidas do rolamento que vêm no próprio dataset (8 esferas, diâmetro da esfera 0,235 e diâmetro do círculo das esferas 1,245) e a velocidade do eixo, dá para calcular quantas pancadas por segundo cada defeito produz. Para enxergar esse ritmo, ficamos com a faixa de 2 a 4 mil ciclos por segundo do sinal (a mesma que a MFPT usa no script de análise dela), tiramos o contorno da vibração (a envoltória) e medimos quanto ele se repete em cada frequência, comparado com o fundo. Esse ritmo vem da geometria do rolamento e não da sessão de gravação, por isso ele é a evidência mais segura de que existe um defeito.

In [ ]:
# Calcula o ritmo de pancadas de cada defeito pela geometria do rolamento e mede, em cada gravação,
# quanto o contorno da vibração se repete nesse ritmo, comparado com o fundo (de 50 a 200 Hz).
from scipy.signal import butter, sosfiltfilt, hilbert, welch

esfera, circulo, n_esferas = 0.235, 1.245, 8
razao = esfera / circulo
ritmo_externa = 0.5 * (1 - razao) * n_esferas
ritmo_interna = 0.5 * (1 + razao) * n_esferas

faixa = butter(4, [2000, 4000], btype='band', fs=taxa, output='sos')

def espectro_envoltoria(sinal):
    envoltoria = np.abs(hilbert(sosfiltfilt(faixa, sinal)))
    return welch(envoltoria - envoltoria.mean(), taxa, nperseg=2 ** 16)

def destaque(freq, potencia, alvo):
    perto = (freq > 0.97 * alvo) & (freq < 1.03 * alvo)
    fundo = np.median(potencia[(freq > 50) & (freq < 200)])
    return potencia[perto].max() / fundo

espectros = gravacoes.sinal.apply(espectro_envoltoria)
gravacoes['externa_x_fundo'] = [destaque(f, p, ritmo_externa * g)
                                for (f, p), g in zip(espectros, gravacoes.giro_hz)]
gravacoes['interna_x_fundo'] = [destaque(f, p, ritmo_interna * g)
                                for (f, p), g in zip(espectros, gravacoes.giro_hz)]

print(f'pista externa: {ritmo_externa * 25:.1f} pancadas por segundo | '
      f'pista interna: {ritmo_interna * 25:.1f} pancadas por segundo (eixo a 25 voltas por segundo)')
gravacoes[['arquivo', 'classe', 'externa_x_fundo', 'interna_x_fundo']].round(1)

In [ ]:
# Mostra o espectro do contorno das mesmas três gravações, com as linhas no ritmo esperado de cada defeito.
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
for ax, nome in zip(axes, exemplos):
    i = gravacoes.index[gravacoes.arquivo == nome][0]
    freq, potencia = espectros[i]
    ax.plot(freq, potencia, lw=0.8)
    ax.axvline(ritmo_externa * gravacoes.giro_hz[i], color='tab:orange', ls='--', label='ritmo da pista externa')
    ax.axvline(ritmo_interna * gravacoes.giro_hz[i], color='tab:red', ls='--', label='ritmo da pista interna')
    ax.set_title(f'{gravacoes.classe[i]} ({nome})', fontsize=10)
    ax.set_ylabel('potência')
axes[0].legend(fontsize=8)
axes[-1].set_xlim(0, 300)
axes[-1].set_xlabel('frequência (pancadas por segundo)')
plt.tight_layout()
plt.show()

### Resultados

Pela geometria do rolamento, com o eixo a 25 voltas por segundo, o defeito na pista externa deveria causar 81,1 pancadas por segundo, e o da pista interna 118,9. É o que aparece. Nas 10 gravações de pista externa, o contorno da vibração se repete no ritmo da pista externa de 420 a 2.380 vezes mais do que o fundo. Nas 7 de pista interna, se repete no ritmo da pista interna de 1.300 a 2.000 vezes mais do que o fundo. Nas 3 normais, os dois ritmos ficam só de 1,6 a 4,1 vezes o fundo, ou seja, não aparecem. Os picos caem em 80,5 e 117,7, menos de 1% abaixo da conta, o que bate com o eixo girando um pouco abaixo de 25 voltas por segundo.

No gráfico do contorno, a pista externa mostra o pico no seu ritmo e um menor no dobro dele. A pista interna mostra o pico no seu ritmo e vários outros espaçados de 25 em 25, a velocidade do eixo: a pista interna gira junto com o eixo, então o defeito passa pela parte mais carregada do rolamento uma vez por volta e as pancadas ficam mais fortes e mais fracas nesse ritmo. Os três gráficos têm escalas diferentes: o pico da pista interna é umas 600 vezes maior que qualquer coisa no normal.

Isso resolve a dúvida da sessão de gravação. A sessão pode mudar a força média ou o deslocamento do sensor, mas não cria uma pancada que se repete exatamente no ritmo previsto pela geometria do rolamento. Vale também para a pista externa, inclusive nas três gravações feitas nas mesmas condições do normal, onde nem a força média nem o pico separavam as classes.

## 3. Janelas e medidas

Para treinar e testar, cortamos cada gravação em janelas de 0,25 s e calculamos dois grupos de medidas para cada janela. As medidas genéricas olham o tamanho e o formato das pancadas: força média (rms), maior pico, curtose (o quanto o sinal é feito de pancadas isoladas) e pico dividido pela força média. As medidas do ritmo olham se o contorno da vibração se repete no ritmo de cada defeito.

Os primeiros 70% de cada gravação vão para o treino e os últimos 30% para o teste. Separamos pelo tempo, e não por gravação, porque só existem 3 gravações normais: separando por gravação, o teste ficaria com uma só.

In [ ]:
# Corta cada gravação em janelas de 0,25 s e calcula as medidas de cada janela. As genéricas
# (força média, pico, curtose e pico dividido pela força média) olham só o tamanho e o formato das
# pancadas; as do ritmo olham se o contorno se repete no ritmo de cada defeito. Os primeiros 70% de
# cada gravação vão para o treino e o resto para o teste, para que toda gravação apareça nos dois.
from scipy.stats import kurtosis

tamanho = int(0.25 * taxa)
genericas = ['rms', 'pico', 'curtose', 'pico_por_rms']
ritmo = ['externa_x_fundo', 'interna_x_fundo']

linhas = []
for _, g in gravacoes.iterrows():
    envoltoria = np.abs(hilbert(sosfiltfilt(faixa, g.sinal)))
    n_janelas = len(g.sinal) // tamanho
    for k in range(n_janelas):
        trecho = slice(k * tamanho, (k + 1) * tamanho)
        s, e = g.sinal[trecho], envoltoria[trecho]
        freq, potencia = welch(e - e.mean(), taxa, nperseg=tamanho)
        rms = np.sqrt(np.mean(s ** 2))
        linhas.append({'arquivo': g.arquivo, 'classe': g.classe, 'carga': g.carga,
                       'parte': 'treino' if k < 0.7 * n_janelas else 'teste',
                       'rms': rms, 'pico': np.abs(s).max(), 'curtose': kurtosis(s),
                       'pico_por_rms': np.abs(s).max() / rms,
                       'externa_x_fundo': destaque(freq, potencia, ritmo_externa * g.giro_hz),
                       'interna_x_fundo': destaque(freq, potencia, ritmo_interna * g.giro_hz)})

janelas = pd.DataFrame(linhas)
janelas.groupby(['classe', 'parte']).size().unstack()

### Resultados

O teste ficou com 21 janelas normais, 42 de pista externa e 21 de pista interna; o treino com 51, 114 e 63.

Numa janela de 0,25 s o ritmo do defeito aparece bem mais fraco do que na gravação inteira (dezenas de vezes o fundo, em vez de centenas ou milhares), porque a janela curta tem menos pancadas para medir. Janelas mais longas deixariam o ritmo mais claro, mas dariam menos janelas normais para treinar.

## 4. Detecção

Na detecção, o modelo aprende apenas com janelas normais do treino e não sabe que tipos de defeito existem; ele só aprende como é o normal. Usamos duas técnicas. O Isolation Forest, que não foi visto em aula, faz cortes ao acaso nos dados: uma janela que fica isolada com poucos cortes é diferente das demais. A distância com PCA, que vimos em aula, mede quantos desvios-padrão do normal uma janela está. Cada técnica roda duas vezes, uma com cada grupo de medidas.

O limite de alerta é a janela normal do treino mais estranha: no teste, o que passar dela é anomalia. Escolhemos esse limite antes de olhar o teste, para não ajustar o resultado aos dados de teste.

In [ ]:
# Detecção: os dois detectores aprendem só com as janelas normais do treino e não sabem que defeitos
# existem. O Isolation Forest isola as janelas com cortes ao acaso (o que fica isolado rápido é
# estranho); a distância com PCA mede quantos desvios-padrão do normal a janela está. O limite de
# cada um é a janela normal de treino mais estranha: o que passar dela é anomalia.
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest

normal_treino = janelas[(janelas.classe == 'normal') & (janelas.parte == 'treino')]
teste = janelas[janelas.parte == 'teste'].copy()
teste['defeito'] = teste.classe != 'normal'

def estranheza_isolation(colunas):
    modelo = IsolationForest(random_state=42).fit(normal_treino[colunas])
    return -modelo.score_samples(normal_treino[colunas]), -modelo.score_samples(teste[colunas])

def estranheza_pca(colunas):
    scaler = StandardScaler().fit(normal_treino[colunas])
    pca = PCA(whiten=True).fit(scaler.transform(normal_treino[colunas]))
    distancia = lambda X: np.linalg.norm(pca.transform(scaler.transform(X)), axis=1)
    return distancia(normal_treino[colunas]), distancia(teste[colunas])

resultados = []
for tecnica, estranheza in [('Isolation Forest', estranheza_isolation), ('Distância com PCA', estranheza_pca)]:
    for nome_medidas, colunas in [('genéricas', genericas), ('ritmo do defeito', ritmo)]:
        treino, avaliado = estranheza(colunas)
        alerta = avaliado > treino.max()
        teste[f'{tecnica} | {nome_medidas}'] = alerta
        resultados.append({'técnica': tecnica, 'medidas': nome_medidas,
                           'TPR': alerta[teste.defeito].mean(), 'FPR': alerta[~teste.defeito].mean()})

pd.DataFrame(resultados).round(3)

In [ ]:
# Quanto cada combinação detecta em cada grupo de gravações do teste. O grupo que importa é a pista
# externa com carga 270, gravada nas mesmas condições do normal.
teste['grupo'] = np.where((teste.classe == 'pista externa') & (teste.carga == 270),
                          'pista externa, carga 270', teste.classe)
colunas_alerta = [c for c in teste.columns if ' | ' in c]
teste.groupby('grupo')[colunas_alerta].mean().round(3).T

### Resultados

| Técnica | Medidas | Detecta defeito (TPR) | Alarme falso (FPR) | Pista externa, carga 270 |
|---|---|---|---|---|
| Isolation Forest | genéricas | 0,905 | 0,000 | 0,714 |
| Isolation Forest | ritmo do defeito | 0,032 | 0,000 | 0,000 |
| Distância com PCA | genéricas | 1,000 | 0,095 | 1,000 |
| Distância com PCA | ritmo do defeito | 0,968 | 0,000 | 1,000 |

A distância com PCA funcionou com os dois grupos de medidas. Com as do ritmo, detectou 97% das janelas com defeito sem nenhum alarme falso. As duas janelas que ela deixou passar são de pista interna (gravação com carga 200), em que o ritmo ficou 9 e 11 vezes acima do fundo. É menos do que a janela normal de treino com o ritmo mais alto por acaso, que chegou a 11,6. Com as genéricas, detectou tudo, mas com 2 alarmes falsos em 21 janelas normais. Esperávamos que as medidas genéricas tivessem dificuldade com a pista externa de carga 270, mas, combinando as quatro, a distância com PCA separou essas janelas do normal.

O Isolation Forest falhou com as medidas do ritmo, detectando só 3% dos defeitos. O motivo está em como ele funciona: ele só corta dentro da faixa de valores que viu no treino, então uma janela com ritmo 50 vezes acima do fundo recebe quase a mesma nota de uma janela normal que, por acaso, ficou no ponto mais alto do treino. Como o limite é justamente a janela normal mais estranha, os defeitos quase nunca passam dele. Com as medidas genéricas ele foi melhor (90%, sem alarme falso), mas perdeu 29% das janelas de pista externa com carga 270, o caso mais difícil.

Um limite dessa avaliação: as janelas normais do teste vêm das mesmas 3 gravações normais do treino, só que do trecho final. Por isso o alarme falso medido aqui tende a ser otimista: numa gravação normal nova, de outro dia ou outra montagem, o detector pode estranhar mais. A detecção de defeito não tem esse problema, porque os detectores nunca viram nenhuma gravação com defeito no treino.

## 5. Diagnóstico

No diagnóstico, o modelo aprende com as três classes e precisa dizer qual é o defeito de cada janela. Usamos os seguintes modelos: regressão logística, Knn, árvore de decisão e Random Forest, cada um com os dois grupos de medidas. O melhor modelo é escolhido por validação cruzada no treino, refeito com o treino inteiro e avaliado no teste com a matriz de confusão e o acerto por classe.

A validação cruzada precisa de um cuidado. Cada gravação vira várias janelas seguidas, e duas janelas vizinhas da mesma gravação são quase iguais: mesmo rolamento, mesma carga, mesma montagem. Na validação cruzada comum, as janelas são sorteadas, então quase sempre uma janela cai no treino e a vizinha na avaliação. Assim o modelo pode acertar só por reconhecer a gravação, e não o defeito. Por isso, na validação cruzada, cada gravação fica inteira de um lado só (GroupKFold, com 3 rodadas, para que cada rodada deixe de fora uma das 3 gravações normais). Rodamos as duas versões para medir a diferença.

In [ ]:
# Compara a validação cruzada comum (janelas sorteadas) com a separada por gravação, para cada
# modelo e grupo de medidas. A diferença entre as duas mostra o quanto o modelo acerta só por
# reconhecer a gravação.
import warnings
from sklearn.model_selection import cross_val_score, KFold, GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

warnings.filterwarnings('ignore')

treino = janelas[janelas.parte == 'treino']
teste_diag = janelas[janelas.parte == 'teste']

models = {'Regressão logística': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
          'Knn (k=3)': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3)),
          'Árvore de decisão': DecisionTreeClassifier(random_state=42),
          'Random Forest': RandomForestClassifier(random_state=42)}
grupos_medidas = {'genéricas': genericas, 'ritmo do defeito': ritmo}

cv_results = []
for nome_medidas, colunas in grupos_medidas.items():
    for nome_modelo, model in models.items():
        comum = cross_val_score(model, treino[colunas], treino.classe,
                                cv=KFold(5, shuffle=True, random_state=42), scoring='accuracy').mean()
        por_gravacao = cross_val_score(model, treino[colunas], treino.classe, cv=GroupKFold(3),
                                       groups=treino.arquivo, scoring='accuracy').mean()
        cv_results.append({'medidas': nome_medidas, 'modelo': nome_modelo,
                           'validação comum': comum, 'separando por gravação': por_gravacao,
                           'diferença': por_gravacao - comum})

cv_results = pd.DataFrame(cv_results)
cv_results.round(3)

In [ ]:
# Para cada grupo de medidas, escolhe o melhor modelo pela validação separada por gravação,
# refaz o treino com todas as janelas de treino e avalia no teste. O acerto por classe é a fração
# de janelas de cada classe classificadas certo; a pista externa com carga 270 aparece à parte.
finais = {}
for nome_medidas, colunas in grupos_medidas.items():
    ranking = cv_results[cv_results.medidas == nome_medidas].sort_values('separando por gravação', ascending=False, kind='stable')
    nome_modelo = ranking.modelo.iloc[0]
    model = models[nome_modelo].fit(treino[colunas], treino.classe)
    y_pred = model.predict(teste_diag[colunas])
    finais[nome_medidas] = (nome_modelo, model, y_pred)

    certo = y_pred == teste_diag.classe
    externa_270 = (teste_diag.classe == 'pista externa') & (teste_diag.carga == 270)
    print(f'== medidas {nome_medidas}: {nome_modelo} ==')
    print(classification_report(teste_diag.classe, y_pred, digits=3))
    print('acerto por classe:', certo.groupby(teste_diag.classe).mean().round(3).to_dict())
    print(f'acerto na pista externa com carga 270: {certo[externa_270].mean():.3f}')
    print()

In [ ]:
# Matriz de confusão do modelo final de cada grupo de medidas, com todas as classes.
classes_ordem = ['normal', 'pista externa', 'pista interna']
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (nome_medidas, (nome_modelo, model, y_pred)) in zip(axes, finais.items()):
    cm = confusion_matrix(teste_diag.classe, y_pred, labels=classes_ordem)
    ConfusionMatrixDisplay(cm, display_labels=classes_ordem).plot(ax=ax, colorbar=False)
    ax.set_title(f'{nome_modelo}, medidas {nome_medidas}', fontsize=10)
    ax.set_xlabel('classe prevista')
    ax.set_ylabel('classe real')
plt.tight_layout()
plt.show()

### Resultados

| Medidas | Modelo | Validação comum | Separando por gravação | Diferença |
|---|---|---|---|---|
| genéricas | Regressão logística | 0,947 | 0,856 | -0,091 |
| genéricas | Knn (k=3) | 0,996 | 0,916 | -0,080 |
| genéricas | Árvore de decisão | 1,000 | 0,881 | -0,119 |
| genéricas | Random Forest | 1,000 | 0,902 | -0,098 |
| ritmo do defeito | Regressão logística | 0,987 | 0,986 | -0,001 |
| ritmo do defeito | Knn (k=3) | 0,996 | 0,991 | -0,005 |
| ritmo do defeito | Árvore de decisão | 0,991 | 0,987 | -0,004 |
| ritmo do defeito | Random Forest | 0,991 | 0,991 | 0,000 |

Com as medidas genéricas, a validação comum promete de 95% a 100% de acerto, mas, quando o modelo é avaliado numa gravação que nunca viu, o acerto cai para 86% a 92%. Parte do acerto vinha de reconhecer a gravação, e não o defeito. Com as medidas do ritmo, as duas validações dão praticamente o mesmo resultado (de 98,6% a 99,1%), porque o ritmo do defeito não depende da gravação.

Pela validação separada por gravação, o melhor modelo foi o Knn nos dois grupos de medidas. Com as medidas do ritmo houve empate entre o Knn e o Random Forest (0,991), e ficou o primeiro da lista.

No teste:

| Medidas | Modelo | Acerto geral | Normal | Pista externa | Pista interna | Pista externa, carga 270 |
|---|---|---|---|---|---|---|
| genéricas | Knn (k=3) | 1,000 | 1,000 | 1,000 | 1,000 | 1,000 |
| ritmo do defeito | Knn (k=3) | 0,988 | 1,000 | 1,000 | 0,952 | 1,000 |

Com as medidas do ritmo, o único erro foi uma janela de pista interna (gravação com carga 200) classificada como normal. É a mesma janela fraca que a detecção deixou passar: nela o ritmo da pista interna ficou só 9 vezes acima do fundo.

As medidas genéricas acertaram todas as janelas do teste, mas esse resultado não quer dizer que elas são melhores. O teste é dividido pelo tempo dentro de cada gravação, então as janelas do teste vêm das mesmas gravações que o modelo viu no treino, só que de outro trecho. Escolhemos essa divisão porque só existem 3 gravações normais, e separar por gravação deixaria o teste com uma só. A consequência é que o teste não mede o que acontece numa gravação nova. Para isso, o número mais honesto é a validação separada por gravação: 92% com as medidas genéricas e 99% com as do ritmo. Ou seja, as medidas genéricas acertam o teste porque reconhecem as gravações; numa gravação nova, as do ritmo são mais confiáveis.